# 05 · Pré-processamento

Define o pré-processamento de cada algoritmo, com base nas conclusões da análise exploratória, e testa os transformadores no conjunto de treino. As funções ficam em `scroll_sono.preprocessing` e devolvem um `ColumnTransformer` ainda não ajustado, que na etapa 6 entra num `Pipeline` junto com o modelo. Assim, o ajuste acontece só nos dados de treino de cada fold da validação cruzada.

In [1]:
%load_ext autoreload
%autoreload 2

import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from scroll_sono.config import FEATURE_SETS, RANDOM_STATE, TARGET
from scroll_sono.data import load_train
from scroll_sono.preprocessing import (ALGORITHMS, LOG_COLS, NOMINAL_ENCODINGS, SCALERS, make_preprocessor,
                                      split_columns)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
train = load_train()
print(f"Treino: {train.shape[0]:,} linhas × {train.shape[1]} colunas")

Treino: 6,800 linhas × 18 colunas


## 1. Decisões por algoritmo

| Decisão | KNN | Naive Bayes (Gaussiano) | Naive Bayes (Categórico) | Árvore de Decisão |
|---|---|---|---|---|
| Valores ausentes | Mediana / moda | Mediana / moda | Mediana / moda | Mediana / moda |
| Numéricas | Padronização (z-score) ou min-max | Sem escala | Discretização em 5 faixas de frequências iguais | Sem transformação |
| `log1p` nas assimétricas | Opcional | Opcional | Não se aplica | Não se aplica |
| `chronotype` | Ordinal + a mesma escala das numéricas | Ordinal | Ordinal | Ordinal |
| Nominais | One-hot | One-hot | Códigos inteiros | Códigos inteiros ou one-hot |
| `blue_light_filter_active` | Sem transformação (já é 0/1) | Sem transformação | Sem transformação | Sem transformação |

Justificativas:

- **Imputação:** a base não tem valores ausentes, mas a limpeza transforma valores fora do domínio em `NaN`, e uma nova versão dos dados poderia trazê-los. Por isso, todo transformador começa imputando: **mediana** nas numéricas, que é menos sensível aos outliers e às distribuições assimétricas que a média, e **moda** nas categóricas e na binária. Como a imputação fica dentro do pipeline, a mediana e a moda são calculadas só no treino de cada fold.
- **Escala no KNN:** o KNN usa distâncias, e as escalas dos atributos são muito diferentes (minutos, mg, %, anos). Sem uma escala comum, os atributos com valores maiores dominariam a distância. Os outros algoritmos não usam distância: a árvore compara cada atributo com um limiar, e o Naive Bayes estima uma distribuição por atributo.
- **Padronização ou min-max:** as duas opções têm vantagens. A padronização (z-score) é menos afetada pelos outliers, que são reais e foram mantidos: com min-max, um único valor extremo comprime todos os outros perto de 0 (na cafeína, a mediana é 0 e o máximo é 250 mg). Já o min-max coloca as numéricas no intervalo [0, 1], o mesmo das colunas one-hot, de modo que numéricas e categóricas pesam de forma parecida na distância; com a padronização, as numéricas variam mais e pesam um pouco mais. A escolha é feita na etapa 6, pela validação cruzada.
- **Codificação ordinal do `chronotype`:** as categorias têm ordem natural (`Morning Lark` < `Intermediate` < `Night Owl`), e a EDA mostrou que o déficit de sono cresce nessa ordem.
- **One-hot nas nominais (KNN e Naive Bayes Gaussiano):** um código inteiro faria esses algoritmos tratarem as categorias como se tivessem ordem e distância entre si. O Naive Bayes Categórico usa os códigos só como rótulos (estima uma probabilidade para cada categoria), então para ele basta um código inteiro.
- **Nominais na árvore: códigos inteiros ou one-hot.** A árvore faz cortes do tipo `occupation_type <= 1,5`, então, com códigos inteiros, a ordem das categorias (alfabética, sem significado) influencia as divisões: uma categoria do meio da ordem só é isolada com dois cortes. Com one-hot, cada categoria vira uma coluna 0/1 e pode ser isolada com um único corte, ao custo de mais colunas. A escolha é feita na etapa 6, pela validação cruzada.
- **Dois Naive Bayes:** o Gaussiano supõe que cada atributo segue uma distribuição normal dentro de cada classe, o que não vale para os atributos assimétricos nem para as colunas one-hot (que só valem 0 ou 1). O Categórico evita essa suposição, porque trabalha com faixas de valores. As duas versões também supõem que os atributos são independentes dentro de cada classe, o que a correlação baixa entre eles (EDA) não garante. Por isso, elas entram como comparação empírica: a validação cruzada da etapa 6 mostra se essas aproximações prejudicam o desempenho.
- **`log1p` opcional:** a EDA mostrou assimetria à direita em `bedtime_phone_minutes`, `caffeine_post_5pm_mg` e `sleep_latency_min`. O `log1p` aproxima essas distribuições de uma forma simétrica, o que pode ajudar o KNN e o Naive Bayes Gaussiano. Se ajuda ou não é decidido na etapa 6, pela validação cruzada.
- **Outliers:** são valores reais, dentro do domínio, e são mantidos. O `log1p` reduz a influência deles no KNN e no Naive Bayes Gaussiano.

## 2. Transformação logarítmica

Assimetria (*skewness*) das colunas candidatas no treino, antes e depois do `log1p`. Valores próximos de 0 indicam uma distribuição simétrica.

In [3]:
assimetria = pd.DataFrame({
    "antes": train[LOG_COLS].skew(),
    "depois do log1p": np.log1p(train[LOG_COLS]).skew(),
}).round(2)
assimetria

,antes,depois do log1p
bedtime_phone_minutes,1.04,-0.56
caffeine_post_5pm_mg,1.80,0.48
sleep_latency_min,0.97,-0.23


**Resultado:** o `log1p` reduz a assimetria das três colunas:

- `caffeine_post_5pm_mg`: de 1,80 para 0,48. A cauda de consumos altos é comprimida, mas os 58% de zeros continuam concentrados num único valor.
- `sleep_latency_min`: de 0,97 para −0,23, quase simétrica.
- `bedtime_phone_minutes`: de 1,04 para −0,56. A transformação corrige demais e cria uma leve assimetria à esquerda, por causa dos valores baixos (1 a 10 minutos), que o logaritmo espalha.

Como o efeito é misto, a etapa 6 compara os modelos com e sem `log1p`.

## 3. Discretização para o Naive Bayes Categórico

As numéricas são divididas em 5 faixas de **frequências iguais** (quantis), que lidam melhor com outliers do que faixas de larguras iguais. Os limites são aprendidos no treino:

In [4]:
cols_comportamental = split_columns(FEATURE_SETS["comportamental"])["numericas"]

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    prep = make_preprocessor("categorical_nb", FEATURE_SETS["comportamental"]).fit(train)

kbins = prep.named_transformers_["num"][-1]  # último passo, depois da imputação
faixas = pd.DataFrame({
    "faixas": kbins.n_bins_,
    "limites": [", ".join(f"{v:g}" for v in limites) for limites in kbins.bin_edges_],
}, index=cols_comportamental)
faixas

,faixas,limites
age,5,"18, 24, 29, 35, 44, 65"
bedtime_phone_minutes,5,"1, 27, 43, 60.5, 87, 180"
screen_brightness_pct,5,"10, 38, 50, 60, 72, 100"
caffeine_post_5pm_mg,3,"0, 20, 70, 250"
physical_activity_min,5,"0, 17, 30, 41, 54, 112"


In [5]:
for aviso in avisos:
    print(aviso.message)

Bins whose width are too small (i.e., <= 1e-8) in feature 3 are removed. Consider decreasing the number of bins.


**Resultado:** todas as numéricas ficam com 5 faixas, exceto `caffeine_post_5pm_mg`. Como 58% dos valores são 0, os três primeiros quantis caem todos em 0, e o scikit-learn junta essas faixas repetidas (é o aviso acima). A cafeína fica com 3 faixas: zero, consumo baixo e consumo alto, o que faz sentido para o atributo. O aviso é esperado e vai se repetir na validação cruzada da etapa 6.

## 4. Redução de dimensionalidade

Número de colunas que cada algoritmo recebe depois do pré-processamento:

In [6]:
def preprocessar(algoritmo, cenario, log_transform=False, scaler="standard", nominal_encoding="ordinal", dados=train):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)  # faixas da cafeína (ver seção 3)
        prep = make_preprocessor(algoritmo, FEATURE_SETS[cenario], log_transform, scaler=scaler,
                                 nominal_encoding=nominal_encoding)
        prep.set_output(transform="pandas")
        return prep, prep.fit_transform(dados)


colunas = pd.DataFrame(
    {cenario: {algoritmo: preprocessar(algoritmo, cenario)[1].shape[1] for algoritmo in ALGORITHMS}
     for cenario in FEATURE_SETS}
)
colunas.index.name = "algoritmo"
colunas

,comportamental,completo
algoritmo,,
knn,21,27
gaussian_nb,21,27
categorical_nb,10,16
tree,10,16


**Decisão: nenhuma redução de dimensionalidade.**

- O maior caso tem 27 colunas para 6.800 linhas de treino, uma proporção confortável, mesmo para o KNN.
- A EDA mostrou que os atributos numéricos comportamentais praticamente não são correlacionados entre si (|ρ| ≤ 0,02), então não há redundância a eliminar, e o PCA não concentraria a informação em poucos componentes.
- Componentes do PCA são combinações de todos os atributos, o que tiraria a interpretabilidade da árvore e da análise de importância dos atributos na etapa 7.
- A árvore já faz sua própria seleção de atributos, escolhendo em cada divisão o atributo mais útil.

## 5. Teste dos transformadores

### 5.1 No treino completo

Para cada cenário, algoritmo e opção (`log1p`, no KNN a escala e na árvore a codificação das nominais), o transformador é ajustado no treino e a saída é verificada.

In [7]:
linhas = []
for cenario in FEATURE_SETS:
    for algoritmo in ALGORITHMS:
        for log_transform in (False, True):
            if log_transform and algoritmo in ("categorical_nb", "tree"):
                continue
            for scaler in (SCALERS if algoritmo == "knn" else ["—"]):
                for nominais in (NOMINAL_ENCODINGS if algoritmo == "tree" else ["—"]):
                    _, X = preprocessar(algoritmo, cenario, log_transform,
                                        scaler=scaler if algoritmo == "knn" else "standard",
                                        nominal_encoding=nominais if algoritmo == "tree" else "ordinal")
                    linhas.append({
                        "cenário": cenario, "algoritmo": algoritmo, "log1p": log_transform, "escala": scaler,
                        "nominais": nominais,
                        "linhas": X.shape[0], "colunas": X.shape[1],
                        "valores ausentes": int(X.isna().sum().sum()),
                        "todas numéricas": all(pd.api.types.is_numeric_dtype(t) for t in X.dtypes),
                    })
teste = pd.DataFrame(linhas)
teste

,cenário,algoritmo,log1p,escala,nominais,linhas,colunas,valores ausentes,todas numéricas
0,comportamental,knn,False,standard,—,6800,21,0,True
1,comportamental,knn,False,minmax,—,6800,21,0,True
2,comportamental,knn,True,standard,—,6800,21,0,True
3,comportamental,knn,True,minmax,—,6800,21,0,True
4,comportamental,gaussian_nb,False,—,—,6800,21,0,True
5,comportamental,gaussian_nb,True,—,—,6800,21,0,True
6,comportamental,categorical_nb,False,—,—,6800,10,0,True
7,comportamental,tree,False,—,ordinal,6800,10,0,True
8,comportamental,tree,False,—,onehot,6800,21,0,True
9,completo,knn,False,standard,—,6800,27,0,True


In [8]:
assert (teste["linhas"] == len(train)).all()
assert (teste["valores ausentes"] == 0).all()
assert teste["todas numéricas"].all()
print("Todos os transformadores geram saídas completas e numéricas.")

Todos os transformadores geram saídas completas e numéricas.


### 5.2 Simulando um fold da validação cruzada

Na etapa 6, cada transformador é ajustado em parte do treino e aplicado na parte restante. O teste abaixo simula isso com o primeiro fold de um `StratifiedKFold`: ajusta na parte de treino do fold e transforma a parte de validação.

In [9]:
fold_treino, fold_validacao = next(
    StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(train, train[TARGET])
)
parte_treino, parte_validacao = train.iloc[fold_treino], train.iloc[fold_validacao]

linhas = []
for cenario in FEATURE_SETS:
    for algoritmo in ALGORITHMS:
        prep, X_treino = preprocessar(algoritmo, cenario, dados=parte_treino)
        X_validacao = prep.transform(parte_validacao)
        linhas.append({
            "cenário": cenario, "algoritmo": algoritmo,
            "colunas iguais": list(X_treino.columns) == list(X_validacao.columns),
            "ausentes na validação": int(X_validacao.isna().sum().sum()),
        })
pd.DataFrame(linhas)

,cenário,algoritmo,colunas iguais,ausentes na validação
0,comportamental,knn,True,0
1,comportamental,gaussian_nb,True,0
2,comportamental,categorical_nb,True,0
3,comportamental,tree,True,0
4,completo,knn,True,0
5,completo,gaussian_nb,True,0
6,completo,categorical_nb,True,0
7,completo,tree,True,0


### 5.3 Imputação

A base não tem valores ausentes, então a imputação é testada numa cópia do treino com `NaN` inseridos de propósito em uma coluna numérica, uma nominal, a ordinal e a binária. O transformador é ajustado no treino original e aplicado na cópia.

In [10]:
com_ausentes = train.copy()
linhas_com_nan = com_ausentes.sample(50, random_state=RANDOM_STATE).index
for coluna in ["bedtime_phone_minutes", "occupation_type", "chronotype", "blue_light_filter_active"]:
    com_ausentes.loc[linhas_com_nan, coluna] = np.nan

linhas = []
for cenario in FEATURE_SETS:
    for algoritmo in ALGORITHMS:
        prep, _ = preprocessar(algoritmo, cenario)
        X = prep.transform(com_ausentes)
        linhas.append({"cenário": cenario, "algoritmo": algoritmo,
                       "ausentes na entrada": int(com_ausentes[FEATURE_SETS[cenario]].isna().sum().sum()),
                       "ausentes na saída": int(X.isna().sum().sum())})
pd.DataFrame(linhas)

,cenário,algoritmo,ausentes na entrada,ausentes na saída
0,comportamental,knn,200,0
1,comportamental,gaussian_nb,200,0
2,comportamental,categorical_nb,200,0
3,comportamental,tree,200,0
4,completo,knn,200,0
5,completo,gaussian_nb,200,0
6,completo,categorical_nb,200,0
7,completo,tree,200,0


**Resultado:** os 200 valores ausentes inseridos são preenchidos em todos os casos, e nenhum `NaN` chega aos modelos.

### 5.4 Exemplo de saída

Saída do KNN no cenário comportamental, com `log1p` e padronização. As numéricas e o `chronotype` ficam com média 0 e desvio padrão 1; as nominais viram colunas 0/1.

In [11]:
_, X_knn = preprocessar("knn", "comportamental", log_transform=True)
X_knn.head()

,bedtime_phone_minutes,caffeine_post_5pm_mg,age,screen_brightness_pct,physical_activity_min,chronotype,gender_Female,gender_Male,gender_Non-Binary,occupation_type_Corporate 9-to-5,occupation_type_Freelance / Creative,occupation_type_Healthcare / Shift Worker,occupation_type_Remote Tech,occupation_type_Student,primary_bedtime_app_Instagram / Reddit,primary_bedtime_app_Messaging / Chat,primary_bedtime_app_News / Reading,primary_bedtime_app_Streaming (Netflix/Hulu),primary_bedtime_app_TikTok / Reels,primary_bedtime_app_YouTube,blue_light_filter_active
0,-0.503637,1.150450,-0.458368,0.498675,-0.468453,-0.046606,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1
1,-0.503637,1.725568,0.921900,0.548965,0.347070,-1.412636,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
2,-1.861141,-0.820880,0.576833,-0.507121,-1.379919,1.319424,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1
3,-0.305807,1.150450,-1.148502,0.498675,-1.236003,-0.046606,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
4,-0.421309,-0.820880,-1.148502,-1.110599,-0.276565,-1.412636,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0


In [12]:
X_knn.describe().loc[["mean", "std"]].round(2)

,bedtime_phone_minutes,caffeine_post_5pm_mg,age,screen_brightness_pct,physical_activity_min,chronotype,gender_Female,gender_Male,gender_Non-Binary,occupation_type_Corporate 9-to-5,occupation_type_Freelance / Creative,occupation_type_Healthcare / Shift Worker,occupation_type_Remote Tech,occupation_type_Student,primary_bedtime_app_Instagram / Reddit,primary_bedtime_app_Messaging / Chat,primary_bedtime_app_News / Reading,primary_bedtime_app_Streaming (Netflix/Hulu),primary_bedtime_app_TikTok / Reels,primary_bedtime_app_YouTube,blue_light_filter_active
mean,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.52,0.46,0.03,0.34,0.11,0.12,0.25,0.19,0.19,0.1,0.07,0.15,0.26,0.23,0.47
std,1.0,1.0,1.0,1.0,1.0,1.0,0.50,0.50,0.17,0.47,0.31,0.32,0.43,0.39,0.39,0.3,0.25,0.36,0.44,0.42,0.50


Com min-max, as numéricas e o `chronotype` ficam entre 0 e 1, a mesma faixa das colunas one-hot:

In [13]:
_, X_knn_minmax = preprocessar("knn", "comportamental", log_transform=True, scaler="minmax")
X_knn_minmax.describe().loc[["min", "max"]].round(2)

,bedtime_phone_minutes,caffeine_post_5pm_mg,age,screen_brightness_pct,physical_activity_min,chronotype,gender_Female,gender_Male,gender_Non-Binary,occupation_type_Corporate 9-to-5,occupation_type_Freelance / Creative,occupation_type_Healthcare / Shift Worker,occupation_type_Remote Tech,occupation_type_Student,primary_bedtime_app_Instagram / Reddit,primary_bedtime_app_Messaging / Chat,primary_bedtime_app_News / Reading,primary_bedtime_app_Streaming (Netflix/Hulu),primary_bedtime_app_TikTok / Reels,primary_bedtime_app_YouTube,blue_light_filter_active
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## Resumo e opções para a etapa 6

O pré-processamento de cada algoritmo é montado por `make_preprocessor(algoritmo, colunas_do_cenário, log_transform, scaler=...)`. Todos começam pela imputação (mediana nas numéricas, moda nas demais) e geram saídas completas e numéricas, nos dois cenários, na simulação de um fold e com valores ausentes inseridos.

Opções que ficam para a validação cruzada da etapa 6 decidir:

| Opção | Algoritmos |
|---|---|
| Com ou sem `log1p` nas colunas assimétricas | KNN e Naive Bayes Gaussiano |
| Padronização (z-score) ou min-max | KNN |
| Naive Bayes Gaussiano ou Categórico | Naive Bayes |
| Número de faixas da discretização (`n_bins`) | Naive Bayes Categórico |
| `class_weight="balanced"` | Árvore de Decisão |
| Nominais em códigos inteiros ou one-hot | Árvore de Decisão |